In [1]:

import numpy as np
import pyarrow.parquet as pq
import pickle, os, time, gc
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import cohen_kappa_score

SAVE_DIR   = '/srv/work/models'
IN_V5      = '/srv/work/te_features_v5.parquet'
RF_PKL     = f'{SAVE_DIR}/rf_v51_model.pkl'
ROUTE_KEYS = ['source_agent', 'cloud_provider', 'region']
TARGET_COL = 'label_ventana'
SPLIT_COL  = 'split'
HORIZON    = 2

RF_PARAMS = dict(
    n_estimators     = 300,
    max_features     = 'sqrt',
    min_samples_leaf = 5,
    n_jobs           = 4,
    class_weight     = 'balanced_subsample',
    random_state     = 42,
    verbose          = 0,
)
print(f'RF params D37: {RF_PARAMS}')


RF params D37: {'n_estimators': 300, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'n_jobs': 4, 'class_weight': 'balanced_subsample', 'random_state': 42, 'verbose': 0}


In [2]:

ID_COLS = ROUTE_KEYS + ['window']

print('Cargando te_features_v5.parquet...')
t0 = time.time()
df = pq.read_table(IN_V5).to_pandas()
print(f'  Shape: {df.shape}')

feat_cols_v5 = [c for c in df.columns
                if c not in ID_COLS + [TARGET_COL, SPLIT_COL, 'y']
                and not c.startswith('n_samp')]
print(f'  Features: {len(feat_cols_v5)}')

df['y'] = df.groupby(ROUTE_KEYS)[TARGET_COL].shift(-HORIZON)
df = df.dropna(subset=['y'])
df['y'] = df['y'].astype(int)
print(f'  Post shift: {len(df):,}')

train_mask = df[SPLIT_COL] == 'train'
val_mask   = df[SPLIT_COL] == 'val'
y_train = df.loc[train_mask, 'y'].values.astype(int)
y_val   = df.loc[val_mask,   'y'].values.astype(int)
print(f'  Train: {len(y_train):,}  Val: {len(y_val):,}')
print(f'  y dist train: {dict(zip(*np.unique(y_train, return_counts=True)))}')


Cargando te_features_v5.parquet...
  Shape: (3763917, 109)
  Features: 100
  Post shift: 3,761,847
  Train: 2,125,778  Val: 644,690
  y dist train: {0: 1899347, 1: 153319, 2: 36421, 3: 36691}


In [3]:

ratio_idx = [i for i, c in enumerate(feat_cols_v5) if 'ratio' in c]
other_idx = [i for i, c in enumerate(feat_cols_v5) if 'ratio' not in c]

print('Extrayendo X_train_rf (float32, imputation D17-v5)...')
X_train_rf = df.loc[train_mask, feat_cols_v5].values.astype('float32')
for idx in ratio_idx:
    m = np.isnan(X_train_rf[:, idx])
    if m.any(): X_train_rf[m, idx] = 1.0
for idx in other_idx:
    m = np.isnan(X_train_rf[:, idx])
    if m.any(): X_train_rf[m, idx] = 0.0
gc.collect()

print('Extrayendo X_val_rf (float32)...')
X_val_rf = df.loc[val_mask, feat_cols_v5].values.astype('float32')
for idx in ratio_idx:
    m = np.isnan(X_val_rf[:, idx])
    if m.any(): X_val_rf[m, idx] = 1.0
for idx in other_idx:
    m = np.isnan(X_val_rf[:, idx])
    if m.any(): X_val_rf[m, idx] = 0.0
gc.collect()

del df; gc.collect()

with open(f'{SAVE_DIR}/scaler_v5.pkl', 'rb') as fh:
    scaler_v5 = pickle.load(fh)

X_train_sc = scaler_v5.transform(X_train_rf).astype('float32')
del X_train_rf; gc.collect()
X_val_sc = scaler_v5.transform(X_val_rf).astype('float32')
del X_val_rf; gc.collect()

print(f'  X_train_sc: {X_train_sc.shape}  X_val_sc: {X_val_sc.shape}')


Extrayendo X_train_rf (float32, imputation D17-v5)...
Extrayendo X_val_rf (float32)...
  X_train_sc: (2125778, 100)  X_val_sc: (644690, 100)


In [4]:

print('Entrenando RF v5.1 (D37)...')
t0 = time.time()
rf_v51 = RandomForestClassifier(**RF_PARAMS)
rf_v51.fit(X_train_sc, y_train)
elapsed = time.time() - t0
print(f'  Completado en {elapsed/60:.1f} min')
print(f'  n_estimators={rf_v51.n_estimators}  min_samples_leaf={rf_v51.min_samples_leaf}')
print(f'  class_weight={rf_v51.class_weight}')

with open(RF_PKL, 'wb') as fh:
    pickle.dump(rf_v51, fh)
print(f'  Guardado: {RF_PKL}')
del X_train_sc, y_train; gc.collect()


Entrenando RF v5.1 (D37)...
  Completado en 38.8 min
  n_estimators=300  min_samples_leaf=5
  class_weight=balanced_subsample
  Guardado: /srv/work/models/rf_v51_model.pkl


24

In [5]:

CHUNK = 50_000
n = len(X_val_sc)
proba_val = np.zeros((n, 4), dtype='float32')
for start in range(0, n, CHUNK):
    end = min(start + CHUNK, n)
    proba_val[start:end] = rf_v51.predict_proba(X_val_sc[start:end]).astype('float32')

qwk_val = cohen_kappa_score(y_val, np.argmax(proba_val, axis=1), weights='quadratic')
print(f'QWK val (re-entrenado) : {qwk_val:.4f}')
print(f'QWK val (referencia)   : 0.6503  (LOG-25)')

ref = np.load(f'{SAVE_DIR}/rf_v51_val_proba.npy')
if ref.shape == proba_val.shape:
    print(f'Max diff vs ref proba  : {float(np.max(np.abs(proba_val - ref))):.6f}')
else:
    qwk_ref = cohen_kappa_score(y_val, np.argmax(ref, axis=1), weights='quadratic')
    print(f'Shape mismatch: nuevo={proba_val.shape} ref={ref.shape}')
    print(f'QWK ref (proba file)   : {qwk_ref:.4f}')

print('rf_v51_model.pkl listo para 06 evaluacion test set v7.ipynb')


QWK val (re-entrenado) : 0.6503
QWK val (referencia)   : 0.6503  (LOG-25)
Max diff vs ref proba  : 0.000000
rf_v51_model.pkl listo para 06 evaluacion test set v7.ipynb
